# Phase 2 (Kaggle): full fine-tune of Qwen3.5-4B

The pilot picked **Qwen3.5-4B** (tied with Gemma-4-E4B on quality, but half the memory and parameters, so cheaper to serve).
This notebook trains it on the **full** Banking77 training set (9,003 examples, 2 epochs), then scores it once on the
full 3,080-example test set.

Same recipe as the pilot (lr 2e-4, effective batch 16) except: more data, 2 epochs, and a smaller LoRA **rank 8** (pilot: 16).
The pilot-vs-full table below therefore mixes data size AND rank; it is not a clean rank ablation.

**What's new vs the pilot**
- Trains on **both T4s** (data-parallel via `torchrun`), roughly halving the time. Falls back to one GPU if needed.
- **Checkpoints every 200 steps.** If Kaggle disconnects, re-run the setup cells and the training cell: it resumes.
- Prints a log line every 20 steps instead of a progress bar.

**Before you run:** (1) upload the newest `banking-finetuning-code.zip` as a new version of your code dataset and add it
to this notebook; (2) Accelerator = **GPU T4 x2**; (3) Internet = **On**.

**IMPORTANT: Kaggle deletes everything in `/kaggle/working` when an interactive session ends.** This notebook therefore saves and
offers a download link right after training and again right after evaluation. Click those links immediately. For the safest
run, use **Save Version -> Save & Run All (Commit)**: outputs are then kept under the notebook's Output tab even if you close the browser.

**Expected time:** ~2 h on two T4s (~3.3 h on one), plus ~25 min for the test eval. For a long unattended run use
**Save Version -> Save & Run All (Commit)**: it keeps running if your browser closes and saves the outputs.

In [ ]:
import os, glob, shutil, subprocess, sys, json, time

def find_src():
    hits = glob.glob('/kaggle/input/**/src/schema.py', recursive=True)
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        shutil.unpack_archive(z, '/kaggle/working/_code')
        hits = glob.glob('/kaggle/working/_code/**/src/schema.py', recursive=True)
        if hits:
            return os.path.dirname(os.path.dirname(hits[0]))
    print(subprocess.run('find /kaggle/input -maxdepth 4 | head -40', shell=True, capture_output=True, text=True).stdout)
    raise FileNotFoundError('code dataset not found: add banking-finetuning-code via Add Input')

SRC_ROOT = find_src()
PROJ = '/kaggle/working/Banking_FInetuning'
# Always refresh code/configs from the dataset (keeps data/ and outputs/ already in PROJ, e.g. checkpoints).
shutil.copytree(SRC_ROOT, PROJ, ignore=shutil.ignore_patterns('__pycache__'), dirs_exist_ok=True)
os.chdir(PROJ)
os.makedirs('logs', exist_ok=True)
os.environ['HF_HOME'] = '/kaggle/tmp/hf'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
assert os.path.exists('configs/full/qwen3_5_4b.yaml'), 'Old code dataset: upload the newest banking-finetuning-code.zip as a new version'
print('project at', PROJ, '| code from', SRC_ROOT)

In [ ]:
# Freeze Kaggle's CUDA build of torch so pip can't replace it.
!pip list --format=freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/constraints.txt
!cat /tmp/constraints.txt
!pip install -q -U -c /tmp/constraints.txt transformers peft accelerate bitsandbytes mlflow scikit-learn pyyaml pandas datasets

In [ ]:
# GPU check: must pass before anything else.
import torch, transformers, peft, bitsandbytes
print('torch', torch.__version__, '| built for CUDA:', torch.version.cuda, '| cuda available:', torch.cuda.is_available(),
      '| GPUs:', torch.cuda.device_count())
print('transformers', transformers.__version__, '| peft', peft.__version__, '| bnb', bitsandbytes.__version__)
assert torch.version.cuda is not None, 'CPU-only torch is installed'
assert torch.cuda.is_available(), 'No GPU visible: set Accelerator to GPU T4 x2 in notebook settings, then restart the session'
NGPU = torch.cuda.device_count()

### Fast kernels for Qwen3.5 (recommended)
Without `flash-linear-attention`, Qwen3.5's attention layers run a very slow pure-PyTorch fallback. The pilot's Qwen
training took 65 min vs Gemma's 39 min, probably for this reason. This cell installs it and tells you whether it is present.

In [ ]:
!pip install -q -c /tmp/constraints.txt flash-linear-attention
import importlib.util
print('flash-linear-attention installed:', importlib.util.find_spec('fla') is not None)

In [ ]:
# Helpers
def stream(cmd, log_path=None, env=None):
    """Run a command, stream its output into the notebook (and a log file), raise on failure."""
    print('$', ' '.join(map(str, cmd)))
    e = dict(os.environ, **(env or {}))
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e)
    log = open(log_path, 'a') if log_path else None
    for line in p.stdout:
        if 'it/s' in line or 's/it' in line:      # tqdm noise from model loading
            continue
        print(line, end='')
        if log:
            log.write(line); log.flush()
    rc = p.wait()
    if rc:
        raise RuntimeError(f'command failed (exit {rc}); see output above')

def train_cmd(*extra, nproc=None):
    n = nproc or NGPU
    base = [sys.executable, '-m', 'src.train.qlora', '--config', CFG, *extra]
    if n > 1:
        return [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node', n, '-m', 'src.train.qlora',   # --standalone: pick a free port (29500 can be taken)
                '--config', CFG, *extra]
    return base

CFG = 'configs/full/qwen3_5_4b.yaml'
OUT = 'outputs/full/qwen3_5_4b'

# ---- Saving helpers. Kaggle DELETES /kaggle/working when an interactive session ends, so we save at every stage. ----
from IPython.display import FileLink, display

def save_stage(zip_name, rel_paths):
    # Zip files (paths relative to the project) into /kaggle/working and show a download link.
    here = os.getcwd()
    os.chdir('/kaggle/working')
    try:
        paths = ' '.join(f'Banking_FInetuning/{p}' for p in rel_paths if os.path.exists(f'/kaggle/working/Banking_FInetuning/{p}'))
        subprocess.run(f'rm -f {zip_name}.zip && zip -qr {zip_name}.zip {paths}', shell=True, check=True)
        print(f'{zip_name}.zip = {os.path.getsize(zip_name + ".zip") / 1e6:.1f} MB  <- CLICK TO DOWNLOAD NOW')
        display(FileLink(f'{zip_name}.zip'))
    finally:
        os.chdir(here)

def push_to_hub(folder, repo_name):
    # Optional: private upload to YOUR Hugging Face account, so the result survives a dead session.
    # Needs a Kaggle Secret named HF_TOKEN (Add-ons > Secrets) holding a token with write access. Skips quietly if absent.
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        print('No HF_TOKEN secret found: skipping the Hugging Face upload (the zip download above still works).')
        return
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    repo_id = f"{api.whoami()['name']}/{repo_name}"
    api.create_repo(repo_id, private=True, exist_ok=True)
    api.upload_folder(folder_path=folder, repo_id=repo_id, commit_message='phase 2 adapter')
    print('uploaded (private) ->', f'https://huggingface.co/{repo_id}')

## 1. Data
Builds the splits (train 9,003 / val 1,000 / test 3,080) if they are not there yet.

In [ ]:
if not os.path.exists('data/stats.json'):
    stream([sys.executable, '-m', 'src.data.prepare'])
for s in json.load(open('data/stats.json')):
    if s['split'] in ('train', 'val', 'test'):
        print(s['split'].ljust(8), 'n =', s['n'])

## 2. Smoke test of the 2-GPU launcher (about 5 min)
Trains on 64 rows through the same `torchrun` path as the real run. This checks that data-parallel training works with
4-bit + LoRA before committing to a multi-hour run. Look for: `world_size=2 ... effective batch 16`, a finite loss,
and no errors. If it fails on 2 GPUs, set `NGPU = 1` and re-run this cell: the recipe is identical, just slower.

It then checks whether the fast Qwen kernels are really in use (no "falling back" warning for `chunk_gated_delta_rule`).

In [ ]:
if glob.glob(f'{OUT}/trainer/checkpoint-*'):
    print('Training checkpoints found: SKIPPING the smoke test so they are not deleted. Go to step 3 to resume.')
else:
    shutil.rmtree('outputs/full', ignore_errors=True)
    stream(train_cmd('--limit', 64), log_path='logs/full_smoke.log')
    txt = open('logs/full_smoke.log').read()
    print('\n--- checks ---')
    print('adapter saved        :', os.path.exists(f'{OUT}/adapter/adapter_model.safetensors'))
    print('fast kernels active  :', 'chunk_gated_delta_rule` is falling back' not in txt)
    print('effective batch line :', [l for l in txt.splitlines() if 'effective batch' in l])
    shutil.rmtree('outputs/full', ignore_errors=True)   # discard the smoke artifacts

## 3. Full training
Prints a loss line every 20 steps and an eval-loss line at the end of each epoch. About 1,126 optimizer steps in total
(563 per epoch at effective batch 16). A checkpoint is saved every 200 steps.

**If the session dies:** re-run the setup cells (up to "Helpers"), skip the smoke test, and run this cell again. `--resume`
continues from the latest checkpoint.

In [ ]:
t0 = time.time()
stream(train_cmd('--resume'), log_path='logs/full_train.log')
print(f'training wall time: {(time.time() - t0) / 60:.1f} min')
print(open(f'{OUT}/train_info.json').read())

### SAVE NOW (do not skip)
The adapter is the product of ~2 hours of GPU time. This saves it immediately and gives you a download link.
Click the link before doing anything else. The Hugging Face upload is optional (see the helper above).

In [ ]:
save_stage('phase2_adapter', [f'{OUT}/adapter', f'{OUT}/train_info.json', 'logs/full_train.log', 'data/stats.json'])
push_to_hub(f'{OUT}/adapter', 'banking-triage-qwen3_5_4b-adapter')

## 4. Final evaluation on the full test set
3,080 held-out examples, one GPU, greedy decoding. This is the number that goes in the README. About 20-25 min.
(The test set is only used here, never for any tuning decision.)

In [ ]:
stream([sys.executable, '-m', 'src.eval.hf_eval', '--config', CFG, '--split', 'data/test.jsonl',
        '--adapter', f'{OUT}/adapter', '--tag', 'finetuned_test', '--batch-size', 16],
       log_path='logs/full_eval.log', env={'CUDA_VISIBLE_DEVICES': '0'})

In [ ]:
# Save the evaluation results right away, too.
save_stage('phase2_eval', [f'{OUT}/finetuned_test.json', f'{OUT}/finetuned_test_preds.jsonl', f'{OUT}/train_info.json', 'logs/full_eval.log'])

## 5. Results vs the pilot, and where it still fails

In [ ]:
import collections
from sklearn.metrics import classification_report

full = json.load(open(f'{OUT}/finetuned_test.json'))
PILOT = {'intent_accuracy': 0.8981, 'intent_macro_f1': 0.9029, 'invented_label_rate': 0.0123,
         'urgency_accuracy': 0.9721, 'needs_human_accuracy': 0.9854, 'json_valid_rate': 1.0}
print(f"{'metric':<24}{'pilot (2k ex.)':>16}{'full (9k ex.)':>16}{'change':>10}")
for k, pv in PILOT.items():
    v = full.get(k, float('nan'))
    print(f'{k:<24}{pv:>16.4f}{v:>16.4f}{v - pv:>+10.4f}')
print('\ntrain info:', {k: v for k, v in json.load(open(f'{OUT}/train_info.json')).items() if k in
      ('train_examples', 'train_seconds', 'peak_vram_gb', 'final_val_loss', 'world_size', 'effective_batch')})

preds = [json.loads(l) for l in open(f'{OUT}/finetuned_test_preds.jsonl')]
labels = sorted({p['gold'] for p in preds})
rep = classification_report([p['gold'] for p in preds], [p['pred'] for p in preds], labels=labels,
                            output_dict=True, zero_division=0)
worst = sorted(labels, key=lambda l: rep[l]['f1-score'])[:10]
print('\n10 weakest intents (F1):')
for l in worst:
    print(f"  {rep[l]['f1-score']:.2f}  {l}")
conf = collections.Counter((p['gold'], p['pred']) for p in preds if p['gold'] != p['pred'])
print('\nTop confusions:')
for (g, q), n in conf.most_common(8):
    print(f'  {n:>3}  {g} -> {q}')
invented = [p['pred'] for p in preds if p['pred'] not in set(labels) and p['pred'] != '__invalid__']
print(f'\nInvented labels: {len(invented)} ({len(invented) / len(preds):.2%}), distinct: {len(set(invented))}')
print('examples:', collections.Counter(invented).most_common(5))

In [ ]:
# A few raw errors, to eyeball whether the "mistakes" are really label ambiguity.
import random
random.seed(0)
errs = [p for p in preds if p['gold'] != p['pred']]
for p in random.sample(errs, min(8, len(errs))):
    print(f"gold={p['gold']:<34} pred={p['pred']:<34} | {p['text'][:90]}")

## 6. Save
Packages the adapter (the thing Phase 3 merges and quantizes), metrics, predictions and logs.
Download `phase2_results.zip` from the output panel (or use Save Version) before the session closes.

In [ ]:
!cd /kaggle/working && rm -f phase2_results.zip && zip -qr phase2_results.zip Banking_FInetuning/outputs/full Banking_FInetuning/logs Banking_FInetuning/data/stats.json -x '*/trainer/*' && ls -lh phase2_results.zip